# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Insurance_Policy_Analytics

**Input file:** `05_Insurance_Policies.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("05_Insurance_Policies.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["premium"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of premium.

In [ ]:
t=    policy_id customer_id  ... policy_status policy_start_date
0      P00001       C0237  ...        Active        2025-03-09
1      P00002       C0044  ...        Active        2026-02-12
2      P00003       C0224  ...       Expired        2024-06-18
3      P00004       C0151  ...        Active        2026-02-06
4      P00005       C0130  ...     Cancelled        2024-06-05
..        ...         ...  ...           ...               ...
377    P00378       C0164  ...     Cancelled        2024-03-01
378    P00379       C0168  ...       Expired        2023-04-22
379    P00380       C0147  ...        Active        2025-03-28
380    P00018       C0183  ...        Active        2026-05-06
381    P00052       C0157  ...        Active        2025-01-21

[382 rows x 9 columns]
threshold=df["premium"].quantile(.75)
print(df[df["premium"]>threshold].sort_values("premium",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total premium, and average claim_amount for each policy_id.

In [ ]:
print(df.groupby("policy_id",dropna=False).agg(count=("premium","count"),total=("premium","sum"),avg=("claim_amount","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze premium by policy_id and customer_id; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["policy_id","customer_id"],dropna=False).agg(count=("premium","count"),total=("premium","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["premium"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="policy_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("policy_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["policy_id"]=x["policy_id"].fillna("Unknown"); print(x["policy_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["premium"]=x["premium"].fillna(x["premium"].median()); print(x["premium"].isna().sum())

## Q11. String search

Find records where policy_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["policy_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by premium descending and claim_amount ascending.

In [ ]:
print(df.sort_values(["premium","claim_amount"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on premium using its median as the threshold.

In [ ]:
x=df.copy(); med=x["premium"].median(); x["segment"]=np.where(x["premium"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("policy_id")["premium"].transform("mean"); print(x[["policy_id","premium","group_avg"]].head(25))

## Q15. Group share

Calculate each policy_id's share of total premium.

In [ ]:
g=df.groupby("policy_id",dropna=False)["premium"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average premium by policy_id and customer_id.

In [ ]:
print(pd.pivot_table(df,index="policy_id",columns="customer_id",values="premium",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("policy_id")[["premium","claim_amount"]].mean().reset_index(); print(w.melt(id_vars="policy_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where premium is negative or missing.

In [ ]:
print("Invalid:",df["premium"].isna().sum()+(df["premium"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in premium.

In [ ]:
q1=df["premium"].quantile(.25); q3=df["premium"].quantile(.75); iqr=q3-q1; print(df[df["premium"]>q3+1.5*iqr][["premium"]].head(30))

## Q20. Top group

Find the category in policy_id with the highest total premium.

In [ ]:
print(df.groupby("policy_id")["premium"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in policy_id with the lowest average premium.

In [ ]:
print(df.groupby("policy_id")["premium"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of premium for each policy_id.

In [ ]:
print(df.groupby("policy_id")["premium"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify premium relative to its median.

In [ ]:
x=df.copy(); med=x["premium"].median(); x["classification"]=x["premium"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["policy_id"].dropna().unique())}}; x=df.copy(); x["policy_id_code"]=x["policy_id"].map(codes); print(x[["policy_id","policy_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null premium, non-null policy_id, and values above the median premium.

In [ ]:
m=df["premium"].median(); x=df[df["premium"].notna()&df["policy_id"].notna()&(df["premium"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("05_Insurance_Policies.csv",chunksize=50):
    total += chunk["premium"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final policy_id-level report with count, total premium, average claim_amount, and rank by total premium.

In [ ]:
r=df.groupby("policy_id",dropna=False).agg(records=("premium","count"),total=("premium","sum"),average=("claim_amount","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.